In [1]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import random
import statsmodels.api as sm

import scipy
# from scipy.linalg.diagsvd

from scipy import linalg

SVD and least squares

We first perform SVD for the design matrix, and then calculate the pseudoinverse (Moore–Penrose inverse) from the SVD. The pseudoinverse for a matrix $A$ is defined as $A^{+} = V \Sigma^{+} U^\top$, where the columns of $U$ and $V$ store the left and the right singular vectors for $A$, respectively, and $\Sigma$ is a rectangular diagonal matrix of which the diagonal elements are singular values for the matrix $A$. $\Sigma^{+}$ is obtained by first calculating the reciprocal of non-zero diagonal elements of $\Sigma$ and then transpose the resulting matrix. For example, if $\Sigma = \begin{bmatrix} 4 & 0 & 0 \\ 0 & 2 & 0 \\ 0 & 0 & 1 \\ 0 & 0 & 0\end{bmatrix}$, then $\Sigma^{+}= \begin{bmatrix} 1/4 & 0 & 0 \\ 0 & 1/2 & 0 \\ 0 & 0 & 1/1 \\ 0 & 0 & 0  \end{bmatrix}^\top = \begin{bmatrix} 1/4 & 0 & 0 & 0 \\ 0 & 1/2 & 0 & 0 \\ 0 & 0 & 1 & 0 \end{bmatrix}$.

In [2]:
df = pd.read_csv('Advertising.csv')
# Please change this file path according to where it is stored on your own Google drive

df = df.iloc[:, 1:] # drop the first column which are just row indices

X = np.c_[np.ones((df.shape[0], 1)), df.iloc[:, 0:-1]]
# the design matrix, including the column storing the dummy feature x_0 = 1,
# along with the 3 feature columns: TV, Radio and Newspaper

y = np.array(df.iloc[:,-1]).reshape(-1,1)
# the target value (the Sales)
# the .reshape(-1,1) will convert the numpy array of shape (200, ) to an array of shape (200, 1)
# so that it behaves as a column of a 2D structure (a column of a matrix), rather than a 1D vector
# (this just helps to ensure consistency during matrix operations)

print(f'The shape of the design matrix (including the dummy variable) for the advertising data is {X.shape}.\n')

U, s, V_transposed = np.linalg.svd(X)

print(f'The shape of U is {U.shape}.\nThe shape of s is {s.shape}.\nThe shape of V^T is {V_transposed.shape}.\n')

The shape of the design matrix (including the dummy variable) for the advertising data is (200, 4).

The shape of U is (200, 200).
The shape of s is (4,).
The shape of V^T is (4, 4).



The `np.linalg.svd()` function returns the singular values instead of the matrix $\Sigma$ or $\Sigma^{+}$. To calculate the pseudoinverse for the design matrix, we'll need to construct $\Sigma^{+}$ from these singular values. This can be accomplished by first taking the reciprocal for each of these singular values, and then involking the function `scipy.linalg.diagsvd()`. This will produce a rectangular diagonal matrix of which the diagonal elements are reciprocals of the singular values. Then we transpose this matrix to obtain $\Sigma^{+}$.

In [3]:
Sigma_plus = scipy.linalg.diagsvd(1/s, X.shape[0], X.shape[1]).T
# construct the sigma matrix of the SVD, based on the singular values, and the shape
# of the matrix for which the SVD was performed

print(f'The shape of this Sigma_plus matrix is {Sigma_plus.shape}.\n')

The shape of this Sigma_plus matrix is (4, 200).



The pseudoinverse is then given by the matrix multiplication product $V\Sigma^{+}U^\top$. Note that the `np.linalg.svd()` we used to calculate the SVD does not return $V$, but instead returns $V^\top$, thus we'll need to transpose $V^\top$ again to obtain $V$.

In [4]:
V = V_transposed.T

X_plus = V @ Sigma_plus @ U.T
# The @ sign is a shortcut for matrix multiplication

print(f'The shape of the pseudoinverse is {X_plus.shape}.\n')

The shape of the pseudoinverse is (4, 200).



The coefficients of this linear regression problem can then be solved simply accoridng to $\beta = X^{+} y$

In [5]:
beta = X_plus @ y

print(beta)

[[ 2.93888937e+00]
 [ 4.57646455e-02]
 [ 1.88530017e-01]
 [-1.03749304e-03]]


NumPy comes with a function `np.linalg.pinv()` that calculates the pseudoinverse of a matrix directly. You can check that the pseudoinverse we calculated above using SVD agrees with the result obtained using this function (within numerical precision).

In [6]:
beta_2 = np.linalg.pinv(X) @ y

print(f'The regression coefficients obtained by directly invoking the pinv() function is:\n{beta_2}')

The regression coefficients obtained by directly invoking the pinv() function is:
[[ 2.93888937e+00]
 [ 4.57646455e-02]
 [ 1.88530017e-01]
 [-1.03749304e-03]]


In [7]:
import statsmodels.api as sm
model = sm.OLS(y, X)
res = model.fit()
res.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                      y   R-squared:                       0.897
Model:                            OLS   Adj. R-squared:                  0.896
Method:                 Least Squares   F-statistic:                     570.3
Date:                Tue, 07 Oct 2025   Prob (F-statistic):           1.58e-96
Time:                        01:12:08   Log-Likelihood:                -386.18
No. Observations:                 200   AIC:                             780.4
Df Residuals:                     196   BIC:                             793.6
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          2.9389      0.312      9.422      0.000       2.324       3.554
x1             0.0458      0.001     32.809      0.000       0.043       0.049
x2             0.1885      0.009     21.893      0.000       0.172       0.206
x3            -0.0010      0.006     -0.177      0.860      -0.013       0.011
==============================================================================
Omnibus:                       60.414   Durbin-Watson:                   2.084
Prob(Omnibus):                  0.000   Jarque-Bera (JB):              151.241
Skew:                          -1.327   Prob(JB):                     1.44e-33
Kurtosis:                       6.332   Cond. No.                         454.
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""